# Musical Profile Clustering

## CMOR 438 Final Project: Music Analytics with a From-Scratch Machine Learning Package

This notebook builds a complete clustering workflow to group songs into interpretable musical profiles using the custom `KMeans` implementation in `music_ml`.

## 1) Motivation

Clustering is an unsupervised method: it does not use popularity labels, and instead groups songs by similarity in audio-feature space. The goal is to discover coherent profiles that can be interpreted musically.

## 2) Load Processed Spotify Dataset

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Make src package importable from notebooks/
sys.path.insert(0, str(Path("..").resolve() / "src"))

from music_ml.preprocessing import StandardScaler
from music_ml.unsupervised import KMeans

data_path = Path("../data/processed/spotify_processed.csv")
df = pd.read_csv(data_path)

print(f"Loaded dataset from: {data_path}")
print(f"Shape: {df.shape}")
df.head()

## 3) Define Clustering Features

We use a focused set of audio characteristics to construct the clustering space.

In [ ]:
cluster_cols = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]

missing_cols = [col for col in cluster_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_cols}")

cluster_df = df[cluster_cols].dropna().copy()
X = cluster_df[cluster_cols].to_numpy()

print(f"Rows used for clustering: {len(cluster_df)}")
print(f"Feature matrix shape: {X.shape}")
cluster_df.head()

## 4) Standardize Features

KMeans relies on distance, so all features are standardized before clustering.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Feature scaling complete.")
print("Scaled feature means (approx. 0):")
print(np.round(X_scaled.mean(axis=0), 3))

## 5) Elbow Method (k = 2 to 8)

We compare inertia across candidate cluster counts to choose a reasonable `k`.

In [ ]:
k_values = list(range(2, 9))
inertias = []

for k in k_values:
    model_k = KMeans(n_clusters=k, max_iters=300, tol=1e-4, random_state=42)
    model_k.fit(X_scaled)
    inertias.append(model_k.inertia_)

elbow_df = pd.DataFrame({"k": k_values, "inertia": inertias})

plt.figure(figsize=(7, 4))
plt.plot(k_values, inertias, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Curve (Custom KMeans)")
plt.xticks(k_values)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

elbow_df

## 6) Fit Custom KMeans

Based on the elbow trend, we set `n_clusters = 4` for interpretable musical profiles.

In [ ]:
kmeans = KMeans(n_clusters=4, max_iters=300, tol=1e-4, random_state=42)
kmeans.fit(X_scaled)

labels = kmeans.labels_

print(f"k: 4")
print(f"Iterations: {kmeans.n_iter_}")
print(f"Inertia: {kmeans.inertia_:.4f}")

## 7) Inspect Cluster Centroids

Centroids summarize the average standardized feature values for each cluster. Positive values indicate above-average feature levels; negative values indicate below-average levels.

In [ ]:
centroids_df = pd.DataFrame(kmeans.centroids_, columns=cluster_cols)
centroids_df.index = [f"cluster_{i}" for i in range(len(centroids_df))]

centroids_df

## 8) PCA Visualization (for plotting only)

We use PCA only to project standardized features into two dimensions for visualization. Clustering itself is still performed in the full feature space.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="tab10", alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("KMeans Clusters in PCA Space")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

print(f"Explained variance by first 2 PCs: {pca.explained_variance_ratio_.sum():.3f}")

## 9) Cluster Size Summary

Before interpretation, we check how many songs are assigned to each cluster.

In [ ]:
cluster_size_df = pd.Series(labels).value_counts().sort_index().to_frame(name="n_songs")
cluster_size_df.index = [f"cluster_{i}" for i in cluster_size_df.index]
cluster_size_df

## 10) Interpret Clusters as Musical Profiles

Interpretation is based directly on `centroids_df`, where each centroid value is in standardized units. For each cluster, the most positive features indicate characteristics that are stronger than the dataset average, while the most negative features indicate characteristics that are weaker.

In practice, this means we can write cluster-level findings in concrete terms, for example:
- a cluster with high `energy` and `loudness` but low `acousticness` can be described as dense, amplified tracks with less acoustic character;
- a cluster with high `acousticness` and lower `energy`/`loudness` reflects softer, more acoustic-centered songs;
- a cluster with high `instrumentalness` and low `speechiness` points to tracks with less vocal/spoken content;
- a cluster with higher `speechiness` and `liveness` suggests speech-forward or performance-like recordings.

The table below uses the **actual top-high and top-low features from each row of `centroids_df`** so each musical profile is grounded in the model output rather than predefined labels.

In [ ]:
profile_rows = []
for cluster_name, row in centroids_df.iterrows():
    top_high = row.sort_values(ascending=False).head(3)
    top_low = row.sort_values(ascending=True).head(3)

    profile_rows.append(
        {
            "cluster": cluster_name,
            "highest_features": ", ".join([f"{f} ({v:.2f})" for f, v in top_high.items()]),
            "lowest_features": ", ".join([f"{f} ({v:.2f})" for f, v in top_low.items()]),
            "music_profile_interpretation": (
                f"This cluster is relatively high in {', '.join(top_high.index[:2])} "
                f"and low in {', '.join(top_low.index[:2])}."
            ),
        }
    )

cluster_interpretation_df = pd.DataFrame(profile_rows)
cluster_interpretation_df